# 🤝 Multi-Agent Workflow Systems with Microsoft Agent Framework (C#)

## 📋 Learning Objectives

This notebook demonstrates how to build sophisticated multi-agent systems using the Microsoft Agent Framework in C#. You'll learn to orchestrate multiple specialized agents working together to solve complex problems through structured workflows.

**Multi-Agent Capabilities You'll Build:**
- 👥 **Agent Collaboration**: Multiple agents working together toward common goals
- 🔄 **Workflow Orchestration**: Structured coordination of agent interactions
- 🎭 **Role Specialization**: Agents with distinct personalities and expertise areas
- 📋 **Quality Assurance**: Review and refinement through agent collaboration

## 🎯 Multi-Agent Architecture Concepts

### Core Multi-Agent Principles
- **Division of Labor**: Each agent specializes in specific domain expertise
- **Collaborative Decision Making**: Agents review and refine each other's work
- **Workflow Coordination**: Structured handoffs and communication patterns
- **Quality Enhancement**: Iterative improvement through multi-perspective analysis

### Agent Interaction Patterns
- **Sequential Processing**: Linear workflow with ordered agent participation
- **Peer Review**: Agents validate and improve each other's outputs
- **Hierarchical Structure**: Lead agents coordinating subordinate specialists
- **Consensus Building**: Multiple agents contributing to final decisions

## 🏗️ Technical Architecture

### Workflow System Components
- **Microsoft Agent Framework**: C# implementation with advanced workflow support
- **WorkflowBuilder**: Declarative workflow definition and execution engine
- **Agent Coordination**: Structured communication and handoff mechanisms
- **Event-Driven Processing**: Reactive workflow execution based on agent outputs

### Multi-Agent Process Flow
```
User Request → Agent 1 (Specialist) → Agent 2 (Reviewer) → Quality Check
                ↓                      ↓                    ↓
         Initial Solution → Review & Feedback → Refined Output → Final Result
```

## 🎭 Agent Role Examples

### Hotel Concierge System
This notebook demonstrates a travel recommendation system with specialized roles:

#### 🏨 **Front Desk Agent**
- **Expertise**: Travel recommendations and local knowledge
- **Personality**: Efficient, experienced, concise communication style
- **Responsibilities**: Generate initial travel suggestions and activities

#### 🎩 **Concierge Agent**  
- **Expertise**: Authentic local experiences and quality assessment
- **Personality**: Discerning, focused on non-touristy recommendations
- **Responsibilities**: Review and refine travel suggestions for authenticity

## 🔧 Technical Implementation

### Workflow Architecture
- **Agent Definition**: Specialized instructions and personality configuration
- **Workflow Builder**: Declarative workflow definition with event handling
- **Communication Protocol**: Structured message passing between agents
- **Result Aggregation**: Combining outputs from multiple agent perspectives

### Event-Driven Coordination
- **WorkflowEvent**: Trigger points for agent activation and handoffs
- **OutputEvent**: Structured data exchange between agents
- **Quality Gates**: Validation checkpoints in the workflow process
- **Feedback Loops**: Iterative refinement through agent collaboration

## ⚙️ Prerequisites & Setup

**Required Dependencies:**
- Microsoft.AgentFramework.Core
- Azure.Identity

**Environment Configuration (.env file):**
```env
AZURE_AI_FOUNDRY_MODEL=your_model_deployment_name
AZURE_AI_FOUNDRY_PROJECT_ENDPOINT=your_project_endpoint
```

## 🎨 Multi-Agent Design Patterns

### 1. **Producer-Consumer Pattern**
- Specialized agents generate content for review by others
- Clear handoff points and data exchange protocols
- Quality assurance through independent review
- Iterative refinement and improvement cycles

### 2. **Committee Pattern**
- Multiple agents contributing different perspectives
- Consensus building through structured discussion
- Democratic decision making with weighted opinions
- Conflict resolution and tie-breaking mechanisms

### 3. **Hierarchical Pattern**
- Lead agents coordinating specialist subordinates  
- Clear authority structures and decision flow
- Escalation paths for complex decisions
- Performance monitoring and quality control

### 4. **Pipeline Pattern**
- Sequential processing with specialized stages
- Each agent adds value in their domain of expertise
- Efficient throughput through parallel processing
- Error handling and recovery at each stage

## 🚀 Advanced Multi-Agent Features

### Workflow Orchestration
- **Dynamic Routing**: Context-based agent selection and routing
- **Parallel Processing**: Concurrent agent execution for efficiency
- **Error Recovery**: Graceful handling of agent failures and retries
- **Performance Monitoring**: Tracking workflow execution and optimization

### Agent Communication
- **Structured Messaging**: Type-safe communication protocols
- **Context Preservation**: Maintaining conversation history across agents
- **Metadata Passing**: Rich information exchange beyond text content
- **Event Broadcasting**: Publish-subscribe patterns for coordination

### Quality Assurance
- **Multi-Perspective Review**: Different agents bringing unique viewpoints
- **Iterative Refinement**: Progressive improvement through collaboration
- **Validation Checkpoints**: Quality gates throughout the workflow
- **Performance Metrics**: Measuring collaboration effectiveness

## 📊 Use Cases & Applications

### Business Process Automation
- Document review and approval workflows
- Customer service escalation systems
- Quality assurance and compliance checking
- Multi-stage content creation and editing

### Research & Analysis
- Peer review systems for research papers
- Multi-analyst financial analysis
- Collaborative report writing and fact-checking
- Academic paper review and improvement

### Creative Collaboration
- Content creation with editors and reviewers
- Multi-perspective creative brainstorming
- Iterative design and feedback systems
- Collaborative storytelling and world-building

Ready to orchestrate intelligent multi-agent collaborations? Let's build systems where agents work together like a high-performing team! 🌟🤖

In [ ]:
#r "nuget: Microsoft.AgentFramework.Core, *-*"
#r "nuget: Azure.Identity, 1.13.1"

In [ ]:
// 📦 Import Standard Libraries
using System;
using System.Collections.Generic;
using System.Linq;
using System.Threading.Tasks;

In [ ]:
// 🤖 Import Microsoft Agent Framework Components for Multi-Agent Workflows
using Azure.Identity;
using Microsoft.AgentFramework;
using Microsoft.AgentFramework.Workflow;

In [ ]:
// 🔧 Load Multi-Agent Workflow Configuration
var modelDeploymentName = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_MODEL");
var projectEndpoint = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_PROJECT_ENDPOINT");

if (string.IsNullOrEmpty(modelDeploymentName) || string.IsNullOrEmpty(projectEndpoint))
{
    throw new InvalidOperationException("Required environment variables are not set. Please check your .env file.");
}

In [ ]:
// 🔗 Initialize Shared Chat Client for Multi-Agent Communication
// Create a unified client that all agents in the workflow will use
// This ensures consistent API access and efficient resource utilization
var credential = new AzureCliCredential();

var chatClient = new AzureAIAgentClient(
    credential: credential,
    modelDeploymentName: modelDeploymentName,
    projectEndpoint: new Uri(projectEndpoint)
);

In [ ]:
// 🎩 Agent 1: Hotel Concierge - Quality Reviewer Role
// This agent specializes in evaluating travel recommendations for authenticity
// Acts as the reviewer stage in our multi-agent workflow for quality assurance
const string REVIEWER_NAME = "Concierge";
const string REVIEWER_INSTRUCTIONS = @"
You are an experienced hotel concierge who has strong opinions about providing the most local and authentic experiences for travelers.

Your role in this multi-agent workflow:
- Review travel recommendations from the Front Desk agent
- Assess whether suggestions provide authentic, non-touristy experiences
- Approve recommendations that meet high standards for local authenticity
- Provide constructive feedback for refinement without giving specific examples

Always focus on the quality and authenticity of experiences rather than just popular tourist destinations.
";

In [ ]:
// 🏨 Agent 2: Front Desk - Initial Recommendations
// This agent generates initial travel suggestions with efficiency
// Acts as the first stage in our multi-agent workflow
const string FRONTDESK_NAME = "FrontDesk";
const string FRONTDESK_INSTRUCTIONS = @"
You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
The goal is to provide the best activities and locations for a traveler to visit.
Only provide a single recommendation per response.
You're laser focused on the goal at hand.
Don't waste time with chit chat.
Consider suggestions when refining an idea.
";

In [ ]:
// 🤖 Create Specialized Agents for Multi-Agent Workflow
var writerAgent = new ChatAgent(
    name: REVIEWER_NAME,
    chatClient: chatClient,
    instructions: REVIEWER_INSTRUCTIONS
);

var reviewerAgent = new ChatAgent(
    name: FRONTDESK_NAME,
    chatClient: chatClient,
    instructions: FRONTDESK_INSTRUCTIONS
);

In [ ]:
// 🔄 Build Multi-Agent Workflow with Sequential Processing
// Define workflow: Front Desk (writer) → Concierge (reviewer)
var workflow = new WorkflowBuilder()
    .SetStartAgent(writerAgent)      // Start with writer agent
    .AddEdge(writerAgent, reviewerAgent)  // Connect writer to reviewer
    .Build();

Console.WriteLine("Multi-agent workflow created with sequential processing pattern.");

In [ ]:
// 🚀 Execute Multi-Agent Workflow with Streaming
var userRequest = "I would like to go to Paris.";
Console.WriteLine($"\n🧳 User Request: {userRequest}\n");
Console.WriteLine("--- Multi-Agent Workflow Execution ---\n");

await foreach (var workflowEvent in workflow.RunStreamingAsync(userRequest))
{
    if (workflowEvent is WorkflowOutputEvent outputEvent)
    {
        Console.WriteLine($"\n✅ Workflow Output from {outputEvent.AgentName}:");
        Console.WriteLine($"{outputEvent.Content}\n");
    }
    else if (workflowEvent is WorkflowAgentStartEvent startEvent)
    {
        Console.WriteLine($"\n▶️ Agent '{startEvent.AgentName}' starting...");
    }
    else if (workflowEvent is WorkflowAgentEndEvent endEvent)
    {
        Console.WriteLine($"◀️ Agent '{endEvent.AgentName}' completed.");
    }
}

Console.WriteLine("\n--- Workflow Execution Complete ---");